# 🛳️ Vessel Path Simulation

This notebook:
1. **Loads** the raw AIS CSV (`AIS_178749379871876535_...csv`)
2. **Builds** per-vessel trajectory feature arrays using the 9-feature engineering pipeline
3. **Runs** the trained Bi-LSTM to predict future positions
4. **Compares** predicted vs actual GPS paths and computes per-step Haversine errors
5. **Exports** a `vessel_simulation_index.json` for the frontend to query
6. **Plots** actual vs predicted trajectories on a map

This enables live simulation mode in the AegisOcean frontend dashboard.

## Cell 1 — Imports & Configuration

In [ ]:
import sys
import math
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Add parent ml/ to path so we can import project modules
ML_DIR = Path('..').resolve()
sys.path.insert(0, str(ML_DIR))

# Load config
with open(ML_DIR / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

# Paths
CSV_PATH      = Path('../../data/AIS_178749379871876535_6390-1787530591097.csv')
RESULTS_DIR   = ML_DIR / CFG['paths']['results_dir']
CKPT_PATH     = RESULTS_DIR / CFG['stage3']['checkpoint_name']
OUTPUT_JSON   = ML_DIR / 'results' / 'vessel_simulation_index.json'

# Device
DEVICE = (
    torch.device('mps')  if torch.backends.mps.is_available()  else
    torch.device('cuda') if torch.cuda.is_available()            else
    torch.device('cpu')
)

# Simulation parameters
MIN_PINGS   = 50    # vessels with fewer pings are excluded
MAX_VESSELS = 30    # max vessels to index for the frontend
T_IN        = CFG['stage3']['t_in']    # 32 input steps
T_OUT       = CFG['stage3']['t_out']   # 8 prediction steps

print(f'Device:      {DEVICE}')
print(f'Checkpoint:  {CKPT_PATH}  (exists={CKPT_PATH.exists()})')
print(f'CSV:         {CSV_PATH}  (exists={CSV_PATH.exists()})')
print(f'T_in={T_IN}  T_out={T_OUT}')

## Cell 2 — Load & Parse the AIS CSV

In [ ]:
# AIS column names in the new file (NOAA format)
AIS_COLS = {
    'MMSI':         'mmsi',
    'BaseDateTime': 'base_date_time',
    'LAT':          'latitude',
    'LON':          'longitude',
    'SOG':          'sog',
    'COG':          'cog',
    'Heading':      'heading',
    'VesselName':   'vessel_name',
    'VesselType':   'vessel_type',
    'Status':       'status',
}

print(f'Loading CSV: {CSV_PATH} ...')
df_raw = pd.read_csv(
    CSV_PATH,
    usecols=list(AIS_COLS.keys()),
    dtype={'MMSI': str, 'VesselName': str, 'VesselType': str},
    parse_dates=['BaseDateTime'],
    low_memory=False,
)

# Rename
df_raw = df_raw.rename(columns=AIS_COLS)

# Drop invalid rows
df_raw = df_raw.dropna(subset=['latitude', 'longitude', 'mmsi', 'base_date_time'])
df_raw['sog'] = pd.to_numeric(df_raw['sog'], errors='coerce').fillna(0.0)
df_raw['cog'] = pd.to_numeric(df_raw['cog'], errors='coerce').fillna(0.0)
df_raw['vessel_type'] = pd.to_numeric(df_raw['vessel_type'], errors='coerce').fillna(0).astype(int)

print(f'Total rows loaded:    {len(df_raw):,}')
print(f'Unique vessels:       {df_raw["mmsi"].nunique():,}')
print(f'Date range:           {df_raw["base_date_time"].min()} → {df_raw["base_date_time"].max()}')
df_raw.head(3)

## Cell 3 — Select Top Vessels by Ping Count

In [ ]:
# Count pings per vessel
ping_counts = df_raw.groupby('mmsi').size().reset_index(name='ping_count')
ping_counts = ping_counts[ping_counts['ping_count'] >= MIN_PINGS]
ping_counts = ping_counts.sort_values('ping_count', ascending=False)

print(f'Vessels with ≥{MIN_PINGS} pings: {len(ping_counts):,}')
print(f'Top 10 vessels by ping count:')
print(ping_counts.head(10).to_string(index=False))

# Select top MAX_VESSELS
top_mmsis = ping_counts.head(MAX_VESSELS)['mmsi'].tolist()
print(f'\nUsing top {len(top_mmsis)} vessels for simulation index')

## Cell 4 — Build 9-Feature Trajectory Arrays

In [ ]:
from ais_dataset import build_trajectory_features, VESSEL_TYPE_RISK, DEFAULT_RISK

trajectories = {}   # { mmsi: np.ndarray (N, 9) }
vessel_meta  = {}   # { mmsi: {name, type_code, type_name, ping_count} }

VESSEL_TYPE_NAMES = {
    range(80, 90): 'Tanker',
    range(70, 80): 'Cargo',
    range(60, 70): 'Passenger',
    range(30, 31): 'Fishing',
    range(50, 60): 'Special Craft',
}

def get_type_name(vtype_code: int) -> str:
    for r, name in VESSEL_TYPE_NAMES.items():
        if vtype_code in r:
            return name
    return 'Other'

for mmsi in top_mmsis:
    vessel_df = df_raw[df_raw['mmsi'] == mmsi].sort_values('base_date_time').reset_index(drop=True)
    
    vtype_code = int(vessel_df['vessel_type'].iloc[0]) if pd.notna(vessel_df['vessel_type'].iloc[0]) else 0
    vname = str(vessel_df['vessel_name'].iloc[0]) if 'vessel_name' in vessel_df.columns else 'Unknown'
    
    # build_trajectory_features expects specific columns — rename to match
    feats = build_trajectory_features(vessel_df)
    
    trajectories[mmsi] = feats
    vessel_meta[mmsi] = {
        'vessel_name':  vname,
        'vessel_type_code': vtype_code,
        'vessel_type':  get_type_name(vtype_code),
        'risk_weight':  VESSEL_TYPE_RISK.get(vtype_code, DEFAULT_RISK),
        'ping_count':   len(vessel_df),
        'start_lat':    float(vessel_df['latitude'].iloc[0]),
        'start_lon':    float(vessel_df['longitude'].iloc[0]),
    }

print(f'Built feature arrays for {len(trajectories)} vessels')
for mmsi, meta in list(vessel_meta.items())[:5]:
    print(f'  MMSI {mmsi}: {meta["vessel_name"]}  ({meta["vessel_type"]})  pings={meta["ping_count"]}  shape={trajectories[mmsi].shape}')

## Cell 5 — Load Trained Bi-LSTM Checkpoint

In [ ]:
from ais_model import build_ais_model

model = build_ais_model(CFG)

if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'✅  Checkpoint loaded — epoch {ckpt.get("epoch","?")} | '
          f'val_hav={ckpt.get("val_hav_km", 0):.3f} km')
else:
    print('⚠️  Checkpoint NOT found — using random weights (predictions will be noisy)')

model.eval().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

## Cell 6 — Haversine Error Helper

In [ ]:
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(max(0.0, a)))

print('haversine_km defined')

## Cell 7 — Run Bi-LSTM Simulation on All Indexed Vessels

In [ ]:
MAX_SOG = 30.0   # knots — matches ais_dataset.py

simulation_results = {}  # { mmsi: { actual_track, predicted_track, step_errors_km } }

for mmsi in top_mmsis:
    feats = trajectories[mmsi]   # (N, 9)
    
    # Need at least T_IN + T_OUT steps to do a predict-then-compare
    min_len = T_IN + T_OUT
    if len(feats) < min_len:
        print(f'  Skip {mmsi}: only {len(feats)} pings (need {min_len})')
        continue

    # Take a sliding window starting at a random stable mid-point
    # (avoid start where vessel may be anchored)
    margin = len(feats) // 3
    start  = random.randint(margin, len(feats) - min_len)
    
    src_np  = feats[start       : start + T_IN]       # (T_IN, 9)
    tgt_np  = feats[start + T_IN: start + min_len]    # (T_OUT, 9)

    # Ground-truth absolute coords
    vessel_df = df_raw[df_raw['mmsi'] == mmsi].sort_values('base_date_time').reset_index(drop=True)
    anchor_lat = float(vessel_df['latitude'].iloc[start + T_IN - 1])
    anchor_lon = float(vessel_df['longitude'].iloc[start + T_IN - 1])

    # Actual T_OUT positions (absolute)
    actual_lats = vessel_df['latitude'].iloc[start + T_IN : start + min_len].tolist()
    actual_lons = vessel_df['longitude'].iloc[start + T_IN : start + min_len].tolist()
    actual_sogs = vessel_df['sog'].iloc[start + T_IN : start + min_len].tolist()

    # Model inference
    src_t = torch.from_numpy(src_np.copy()).unsqueeze(0).to(DEVICE)  # (1, T_IN, 9)
    with torch.no_grad():
        pred_np = model.predict(src_t, t_out=T_OUT).squeeze(0).cpu().numpy()  # (T_OUT, 3)

    # Decode predictions → absolute coords
    pred_lats, pred_lons, pred_sogs = [], [], []
    for i in range(T_OUT):
        pred_lats.append(round(anchor_lat + float(pred_np[i, 0]), 5))
        pred_lons.append(round(anchor_lon + float(pred_np[i, 1]), 5))
        pred_sogs.append(round(float(pred_np[i, 2]) * MAX_SOG, 2))

    # Step-by-step Haversine errors
    step_errors = []
    for i in range(T_OUT):
        err = haversine_km(actual_lats[i], actual_lons[i], pred_lats[i], pred_lons[i])
        step_errors.append(round(err, 3))

    # Input track (historical) for visualisation
    hist_lats = vessel_df['latitude'].iloc[start : start + T_IN].tolist()
    hist_lons = vessel_df['longitude'].iloc[start : start + T_IN].tolist()

    simulation_results[mmsi] = {
        'mmsi':             mmsi,
        'vessel_name':      vessel_meta[mmsi]['vessel_name'],
        'vessel_type':      vessel_meta[mmsi]['vessel_type'],
        'risk_weight':      vessel_meta[mmsi]['risk_weight'],
        'ping_count':       vessel_meta[mmsi]['ping_count'],
        'history_track':    [[lon, lat] for lat, lon in zip(hist_lats, hist_lons)],
        'actual_track':     [[lon, lat] for lat, lon in zip(actual_lats, actual_lons)],
        'predicted_track':  [[lon, lat] for lat, lon in zip(pred_lats, pred_lons)],
        'actual_sogs':      [round(s, 1) for s in actual_sogs],
        'predicted_sogs':   pred_sogs,
        'step_errors_km':   step_errors,
        'mean_error_km':    round(float(np.mean(step_errors)), 3),
        'anchor_lat':       anchor_lat,
        'anchor_lon':       anchor_lon,
    }

print(f'\nSimulation complete: {len(simulation_results)} vessels processed')
for mmsi, r in list(simulation_results.items())[:5]:
    print(f'  {r["vessel_name"]:30s}  mean_err={r["mean_error_km"]:.3f} km  steps={len(r["step_errors_km"])}')

## Cell 8 — Plot: Actual vs Predicted Paths

In [ ]:
# Pick the first 4 vessels to visualise
plot_mmsis = list(simulation_results.keys())[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#0b1724')

for ax, mmsi in zip(axes.flat, plot_mmsis):
    r = simulation_results[mmsi]
    ax.set_facecolor('#0b1724')

    # History
    hist = r['history_track']
    ax.plot([c[0] for c in hist], [c[1] for c in hist],
            color='#4a6fa5', linewidth=1.2, alpha=0.6, label='History (AIS)')

    # Actual future track
    act = r['actual_track']
    ax.plot([c[0] for c in act], [c[1] for c in act],
            color='#00f2fe', linewidth=2.5, marker='o', markersize=5, label='Actual future')

    # Predicted future track
    pred = r['predicted_track']
    ax.plot([c[0] for c in pred], [c[1] for c in pred],
            color='#b877ff', linewidth=2.5, linestyle='--', marker='x', markersize=6, label='Bi-LSTM prediction')

    # Anchor point
    ax.scatter([r['anchor_lon']], [r['anchor_lat']], color='#f983e9', s=80, zorder=5, label='Prediction anchor')

    ax.set_title(f"{r['vessel_name']}  |  {r['vessel_type']}\n"
                 f"Mean error: {r['mean_error_km']:.3f} km",
                 color='#c6f1f7', fontsize=10)
    ax.tick_params(colors='#7a9bc0')
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f°E'))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f°N'))
    for spine in ax.spines.values():
        spine.set_color('#1e3352')
    ax.legend(loc='upper left', fontsize=7, facecolor='#0b1724', labelcolor='white', framealpha=0.8)

plt.suptitle('AegisOcean · Bi-LSTM Vessel Path Simulation\nActual vs Predicted Trajectories',
             color='#c6f1f7', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(ML_DIR / 'results' / 'plots' / 'vessel_simulation_paths.png',
            dpi=150, bbox_inches='tight', facecolor='#0b1724')
plt.show()
print('Plot saved to ml/results/plots/vessel_simulation_paths.png')

## Cell 9 — Plot: Step-Horizon Error Analysis

In [ ]:
# Aggregate step errors across all vessels
all_step_errors = np.array([r['step_errors_km'] for r in simulation_results.values()])  # (N, T_OUT)
mean_step = all_step_errors.mean(axis=0)
p25_step  = np.percentile(all_step_errors, 25, axis=0)
p75_step  = np.percentile(all_step_errors, 75, axis=0)
steps = np.arange(1, T_OUT + 1)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0b1724')
ax.set_facecolor('#0b1724')

ax.fill_between(steps, p25_step, p75_step, alpha=0.25, color='#00f2fe', label='IQR (25-75%)')
ax.plot(steps, mean_step, color='#00f2fe', linewidth=2.5, marker='o', markersize=7, label='Mean error')

# Reference line from training evaluation
eval_means = [0.931, 0.898, 0.910, 0.937, 1.003, 1.061, 1.133, 1.182]
if len(eval_means) >= T_OUT:
    ax.plot(steps, eval_means[:T_OUT], color='#f59e0b', linewidth=1.5, linestyle=':',
            marker='s', markersize=5, label='Published eval (train set)')

ax.set_xlabel('Prediction Step', color='#7a9bc0')
ax.set_ylabel('Mean Haversine Error (km)', color='#7a9bc0')
ax.set_title(f'Bi-LSTM Step-Horizon Error — {len(simulation_results)} vessels from new AIS CSV',
             color='#c6f1f7', fontsize=12)
ax.tick_params(colors='#7a9bc0')
for spine in ax.spines.values():
    spine.set_color('#1e3352')
ax.legend(facecolor='#0b1724', labelcolor='white')
ax.yaxis.grid(True, alpha=0.15, color='white')
ax.set_xticks(steps)

plt.tight_layout()
plt.savefig(ML_DIR / 'results' / 'plots' / 'vessel_simulation_step_errors.png',
            dpi=150, bbox_inches='tight', facecolor='#0b1724')
plt.show()
print(f'Mean errors per step: {[f"{e:.3f}" for e in mean_step.tolist()]}')

## Cell 10 — Export Simulation Index JSON (for Frontend & FastAPI)

In [ ]:
# Build a compact index for the frontend and /ml/simulate endpoint
index_payload = {
    'generated_at':   pd.Timestamp.now().isoformat(),
    'csv_source':     CSV_PATH.name,
    'model_checkpoint': str(CKPT_PATH.name),
    'model_metrics': {
        'published_mean_error_km': 1.007,
        'this_csv_mean_error_km':  round(float(np.mean([r['mean_error_km'] for r in simulation_results.values()])), 3),
    },
    't_in':  T_IN,
    't_out': T_OUT,
    'vessels': [
        {
            'mmsi':            r['mmsi'],
            'vessel_name':     r['vessel_name'],
            'vessel_type':     r['vessel_type'],
            'risk_weight':     r['risk_weight'],
            'ping_count':      r['ping_count'],
            'history_track':   r['history_track'],
            'actual_track':    r['actual_track'],
            'predicted_track': r['predicted_track'],
            'actual_sogs':     r['actual_sogs'],
            'predicted_sogs':  r['predicted_sogs'],
            'step_errors_km':  r['step_errors_km'],
            'mean_error_km':   r['mean_error_km'],
            'anchor_lat':      r['anchor_lat'],
            'anchor_lon':      r['anchor_lon'],
        }
        for r in simulation_results.values()
    ]
}

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, 'w') as f:
    json.dump(index_payload, f, indent=2)

print(f'✅  Saved simulation index → {OUTPUT_JSON}')
print(f'    Vessels indexed:          {len(index_payload["vessels"])}')
print(f'    Overall mean error (km):  {index_payload["model_metrics"]["this_csv_mean_error_km"]}')
print(f'    File size:                {OUTPUT_JSON.stat().st_size / 1024:.1f} KB')

## Cell 11 — Quick Summary Table

In [ ]:
summary_rows = []
for r in simulation_results.values():
    summary_rows.append({
        'MMSI':         r['mmsi'],
        'Vessel Name':  r['vessel_name'],
        'Type':         r['vessel_type'],
        'Pings':        r['ping_count'],
        'Mean Error (km)': r['mean_error_km'],
        'Step 1 (km)':  r['step_errors_km'][0],
        'Step 8 (km)':  r['step_errors_km'][-1],
    })

df_summary = pd.DataFrame(summary_rows).sort_values('Mean Error (km)')
print('Simulation Results Summary:')
print(df_summary.to_string(index=False))